In [82]:
import pandas as pd
import numpy as np
import spacy


In [83]:
import re

def nettoyer_phrase(phrase):
    phrase = phrase.strip()
    
   # uniformiser les tirets
    phrase = phrase.replace("–", "-").replace("—", "-")
    
    # enlever le tiret de dialogue en début
    phrase = re.sub(r"^\-\s*", "", phrase)
    
    # uniformiser les apostrophes
    phrase = phrase.replace("'", "’")
    
    # supprimer les espaces après une apostrophe élidée : s’ y -> s’y
    phrase = re.sub(r"([A-Za-zÀ-ÿ])’\s+([A-Za-zÀ-ÿ])", r"\1’\2", phrase)
    
    # espaces multiples
    phrase = re.sub(r"\s+", " ", phrase)
    
    return phrase.strip()

In [84]:
nlp = spacy.load("fr_core_news_lg")

In [85]:
# 1. Ouvrir le fichier et lire toutes les lignes d'un coup
with open("corpus_goncourt/1860_Charles Demailly_travail.txt", "r", encoding="utf-8") as f:
    lignes = f.readlines()


    

In [86]:
data = []

# 2. Boucler sur chaque ligne (qui est déjà une phrase)
for ligne in lignes:
    phrase_nettoyee = nettoyer_phrase(ligne) # Enlève les espaces et les \n invisibles
    
    if phrase_nettoyee: 
        data.append({
            'nom_fichier': "1860_Charles Demailly",
            'phrase': phrase_nettoyee
        })


df_final = pd.DataFrame(data)


print(f"Nombre de phrases chargées : {len(df_final)}")
df_final.head(10)

Nombre de phrases chargées : 1539


,nom_fichier,phrase
0,1860_Charles Demailly,Un article ?… Tu me demandes s’y a un article ...
1,1860_Charles Demailly,Hein ! la société ?…
2,1860_Charles Demailly,Tu ne la connais pas ? C’est pourtant une soci...
3,1860_Charles Demailly,Après ?
4,1860_Charles Demailly,"Après, après ? tu poses ta femme : une comédie..."
5,1860_Charles Demailly,Ceci était dit dans une grande pièce tendue d’...
6,1860_Charles Demailly,y avait cinq hommes dans cette pièce qui était...
7,1860_Charles Demailly,"L’un avait des cheveux blonds, le front court,..."
8,1860_Charles Demailly,L’autre était un jeune homme de trente-quatre ...
9,1860_Charles Demailly,"Des cheveux de femme, une bouche de femme, un ..."


In [87]:
df_final["phrase"] = df_final["phrase"].astype(str).str.strip()
df_final = df_final[df_final["phrase"] != ""].copy()

In [88]:
def get_pos_pattern(phrase):
    doc = nlp(phrase)
    return " ".join([token.pos_ for token in doc])

df_final["motif_pos"] = df_final["phrase"].apply(get_pos_pattern)

In [89]:
df_final

,nom_fichier,phrase,motif_pos
0,1860_Charles Demailly,Un article ?… Tu me demandes s’y a un article ...,DET NOUN PUNCT PUNCT PRON PRON VERB PRON PRON ...
1,1860_Charles Demailly,Hein ! la société ?…,PROPN PUNCT DET NOUN PUNCT PUNCT
2,1860_Charles Demailly,Tu ne la connais pas ? C’est pourtant une soci...,PRON ADV DET NOUN ADV PUNCT PRON AUX ADV DET N...
3,1860_Charles Demailly,Après ?,ADP PUNCT
4,1860_Charles Demailly,"Après, après ? tu poses ta femme : une comédie...",ADP PUNCT ADP PUNCT PRON VERB DET NOUN PUNCT D...
...,...,...,...
1534,1860_Charles Demailly,Et ne répondit pas.,CCONJ ADV VERB ADV PUNCT
1535,1860_Charles Demailly,Nachette resta huit jours à Saint-Sauveur. fut...,PROPN VERB NUM NOUN ADP PROPN PUNCT AUX ADJ AD...
1536,1860_Charles Demailly,"Bref, Nachette amusa Marthe, désarma Charles, ...",ADV PUNCT PROPN VERB PROPN PUNCT VERB PROPN PU...
1537,1860_Charles Demailly,"Nachette parti, la campagne sembla à Marthe vi...",PROPN ADJ PUNCT DET NOUN VERB ADP PROPN NOUN A...


In [79]:
for phrase in df_final["phrase"].head(5):
    doc = nlp(phrase)
    print("\nPHRASE :", phrase)
    for token in doc:
        print(token.text, "->", token.pos_)


PHRASE : Un article ?… Tu me demandes s’y a un article dans mon histoire ? Mais, malheureux, un enfant de six ans en ferait une comédie en vers, les yeux bandés ! Scène première : le foyer de la Comédie-Française… tu comprends… la maison de Molière… Talma… les souvenirs… la tirade : c’est là où César cause avec Scapin, où Melpomène prend l’éventail de Thalie, où… où… où… n’y a pas de raison pour que ça finisse ! Tu passes aux indiscrétions : Provost et Anselme qui jouent aux échecs, l’ingénue qui demande une glace, l’huissier qui fait découvrir le grand-duc héréditaire de Toscane, et mademoiselle Fix qui le fait rougir de ne s’être point découvert de lui-même, la Société du rachat des captifs…
Un -> DET
article -> NOUN
? -> PUNCT
… -> PUNCT
Tu -> PRON
me -> PRON
demandes -> VERB
s’ -> PRON
y -> PRON
a -> VERB
un -> DET
article -> NOUN
dans -> ADP
mon -> DET
histoire -> NOUN
? -> PUNCT
Mais -> CCONJ
, -> PUNCT
malheureux -> ADJ
, -> PUNCT
un -> DET
enfant -> NOUN
de -> ADP
six -> NUM
a

In [90]:
df_final.to_csv("corpus_goncourt/1860_Charles Demailly_motif_pos.csv", index=False, encoding="utf-8")